In [1]:
import pyautogui
import pandas as pd
import numpy as np
from importlib import reload
from source import select


## Read in telemetry file

### **select file**

In [9]:
reload(select)

ifile = select.single_file()

ifile

'C:/Users/Nortek1000/Desktop/toolbox/data/txt/hval_2025-10-11_37_days.txt'

### **read data into dictionary**

In [10]:
import re
from collections import defaultdict

data = defaultdict(lambda: defaultdict(list))

with open(ifile, 'r') as f:
    for line in f:
        try:
            match = re.match(r'\$(\w+),(.*)\*(\w+)', line.strip())
            if not match:
                continue
            tm_type, content = match.group(1), match.group(2)
            fields = {}
            for field in content.split(','):
                if '=' in field:
                    k, v = field.split('=', 1)
                    fields[k] = v
            for key, value in fields.items():
                data[tm_type][key].append(value)
        except Exception as e:
            print(f"Skipping line due to error: {line.strip()}\n{e}")

data.keys()

dict_keys(['PNORI2', 'PNORS2', 'PNORC2'])

### **check contents**

In [11]:
pd.DataFrame(data['PNORC2'])

,DATE,TIME,CN,CP,VE,VN,VU,VU2,A1,A2,A3,A4,C1,C2,C3,C4
0,101125,084948,1,2.0,-0.023,-0.028,0.003,0.000,70.7,70.2,70.8,70.2,92,95,96,95
1,101125,084948,2,3.0,-0.041,-0.012,-0.005,-0.004,64.5,64.5,64.3,64.5,96,98,96,97
2,101125,084948,3,4.0,-0.033,-0.011,0.004,-0.002,63.2,60.4,59.4,60.8,99,97,98,99
3,101125,084948,4,5.0,-0.032,-0.016,-0.001,-0.004,59.4,58.9,56.6,58.0,96,99,98,99
4,101125,084948,5,6.0,-0.027,-0.035,-0.001,-0.003,55.2,54.5,54.9,55.4,98,95,99,98
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
767218,111725,023948,25,26.0,-32.768,-32.768,-32.768,-32.768,28.8,28.5,28.8,29.7,18,16,21,29
767219,111725,023948,26,27.0,-32.768,-32.768,-32.768,-32.768,30.0,29.4,33.3,42.6,32,28,61,94
767220,111725,023948,27,28.0,-0.154,-0.145,-0.016,-0.004,35.6,37.6,41.7,45.6,80,87,93,97
767221,111725,023948,28,29.0,-0.044,-0.106,-0.005,0.014,36.8,40.4,43.8,43.4,85,92,96,94


## Save and format data

### **convert to xarray.Dataset**

In [12]:
import xarray as xr
import pandas as pd

def parse_time(df):
    df['TIME'] = pd.to_datetime(df['DATE']+' '+df['TIME'], format='%m%d%y %H%M%S')
    return df.drop(columns='DATE')

try:
    # Process PNORC2
    cdf = parse_time(pd.DataFrame(data['PNORC2'])).set_index(['TIME', 'CN'])
    ds = cdf.to_xarray()
    
    # Process PNORS2
    sdf = parse_time(pd.DataFrame(data['PNORS2'])).set_index('TIME')
    for col in sdf.columns:
        ds[col] = ('TIME', sdf[col])
    
    # Add PNORI2
    for k, v in data['PNORI2'].items():
        ds[k] = v[0]

except KeyError as e:
    print(f"Missing expected keys in data: {e}")
    ds = xr.Dataset()

ds

<xarray.Dataset> Size: 83MB
Dimensions:  (TIME: 26456, CN: 29)
Coordinates:
  * TIME     (TIME) datetime64[ns] 212kB 2025-10-11T08:49:48 ... 2025-11-17T0...
  * CN       (CN) object 232B '1' '10' '11' '12' '13' ... '5' '6' '7' '8' '9'
Data variables: (12/33)
    CP       (TIME, CN) object 6MB '2.0' '11.0' '12.0' ... '8.0' '9.0' '10.0'
    VE       (TIME, CN) object 6MB '-0.023' '-0.026' ... '-0.035' '-0.029'
    VN       (TIME, CN) object 6MB '-0.028' '-0.014' '0.002' ... '0.019' '0.034'
    VU       (TIME, CN) object 6MB '0.003' '-0.008' '0.002' ... '0.020' '0.017'
    VU2      (TIME, CN) object 6MB '0.000' '-0.001' '0.002' ... '0.003' '0.006'
    A1       (TIME, CN) object 6MB '70.7' '45.9' '43.2' ... '44.0' '48.8' '46.5'
    ...       ...
    SN       <U6 24B '106516'
    NB       <U1 4B '4'
    NC       <U2 8B '29'
    BD       <U4 16B '1.00'
    CS       <U4 16B '1.00'
    CY       <U3 12B 'ENU'

In [13]:
import numpy as np

def assign_dtypes(ds, dtypes, fill=None):
    for col in ds.variables:
        if col in dtypes.keys():
            ds[col] = ds[col].astype(dtypes[col])
        else:
            if fill: 
                ds[col] = ds[col].astype(fill)

dtypes = dict(
    TIME = np.datetime64,
    CN = int,
    EC = int,
    IT = int,
    NB = int,
    NC = int,
    SC = str,
    SN = str,
    CY = str
)

assign_dtypes(ds, dtypes, fill=float)

ds

<xarray.Dataset> Size: 83MB
Dimensions:  (TIME: 26456, CN: 29)
Coordinates:
  * TIME     (TIME) datetime64[ns] 212kB 2025-10-11T08:49:48 ... 2025-11-17T0...
  * CN       (CN) int64 232B 1 10 11 12 13 14 15 16 17 ... 28 29 3 4 5 6 7 8 9
Data variables: (12/33)
    CP       (TIME, CN) float64 6MB 2.0 11.0 12.0 13.0 14.0 ... 7.0 8.0 9.0 10.0
    VE       (TIME, CN) float64 6MB -0.023 -0.026 -0.035 ... -0.035 -0.029
    VN       (TIME, CN) float64 6MB -0.028 -0.014 0.002 ... -0.029 0.019 0.034
    VU       (TIME, CN) float64 6MB 0.003 -0.008 0.002 0.0 ... 0.001 0.02 0.017
    VU2      (TIME, CN) float64 6MB 0.0 -0.001 0.002 0.006 ... 0.005 0.003 0.006
    A1       (TIME, CN) float64 6MB 70.7 45.9 43.2 40.5 ... 60.2 44.0 48.8 46.5
    ...       ...
    SN       <U6 24B '106516'
    NB       int64 8B 4
    NC       int64 8B 29
    BD       float64 8B 1.0
    CS       float64 8B 1.0
    CY       <U3 12B 'ENU'

In [14]:
print(ds)

<xarray.Dataset> Size: 83MB
Dimensions:  (TIME: 26456, CN: 29)
Coordinates:
  * TIME     (TIME) datetime64[ns] 212kB 2025-10-11T08:49:48 ... 2025-11-17T0...
  * CN       (CN) int64 232B 1 10 11 12 13 14 15 16 17 ... 28 29 3 4 5 6 7 8 9
Data variables: (12/33)
    CP       (TIME, CN) float64 6MB 2.0 11.0 12.0 13.0 14.0 ... 7.0 8.0 9.0 10.0
    VE       (TIME, CN) float64 6MB -0.023 -0.026 -0.035 ... -0.035 -0.029
    VN       (TIME, CN) float64 6MB -0.028 -0.014 0.002 ... -0.029 0.019 0.034
    VU       (TIME, CN) float64 6MB 0.003 -0.008 0.002 0.0 ... 0.001 0.02 0.017
    VU2      (TIME, CN) float64 6MB 0.0 -0.001 0.002 0.006 ... 0.005 0.003 0.006
    A1       (TIME, CN) float64 6MB 70.7 45.9 43.2 40.5 ... 60.2 44.0 48.8 46.5
    ...       ...
    SN       <U6 24B '106516'
    NB       int64 8B 4
    NC       int64 8B 29
    BD       float64 8B 1.0
    CS       float64 8B 1.0
    CY       <U3 12B 'ENU'


### **export to netCDF**

In [15]:
import os

filename = os.path.basename(ifile).rpartition('.')[0]
ofile = f'data/netcdf/{filename}.nc'

ds.to_netcdf(ofile)